# 第6章 シーンと可視化

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第6章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

scene.add(Receiver('rx0', position=[60.0, -40.0, 1.5]))
paths = solver(scene, max_depth=5)

rm = rmsolver(scene, max_depth=3, cell_size=(5.0, 5.0),
              samples_per_tx=10**6)

# アニメーションの節で使う受信機の軌跡と経路長
trajectory = lambda t: [10.0 * t - 200.0, 50.0, 1.5]
s = 0.0

## 2. 組み込みシーンを使う

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

available = [s for s in dir(scene_assets) if not s.startswith('_')]
print('Available scenes:', available[:10])

scene = load_scene(scene_assets.munich)

## 3. カメラを設定する

In [ ]:
from sionna.rt import Camera

cam = Camera(position=[-300, 300, 150],
             look_at=[0, 0, 0])

## 4. シーンをプレビューする

In [ ]:
scene.preview()

## 5. 斜め視点でレンダリングする

In [ ]:
img = scene.render(camera=cam,
                   resolution=[1280, 720],
                   num_samples=128)

## 6. パスを表示する

In [ ]:
img = scene.render(camera=cam, paths=paths,
                   resolution=[1280, 720], num_samples=64)

## 7. 電波マップを重ねて表示する

In [ ]:
img = scene.render(camera=cam, radio_map=rm,
                   resolution=[1280, 720])

## 8. matplotlib で結果を描く

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
rss_dbm = 10*np.log10(rm.rss.numpy()[0]) + 30
plt.imshow(rss_dbm, origin='lower', cmap='jet', vmin=-120, vmax=-40)
plt.colorbar(label='RSS [dBm]')
plt.xlabel('x [cells]'); plt.ylabel('y [cells]')

## 9. 読みやすい図を作る

In [ ]:
plt.rcParams['font.size'] = 11
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(rss_dbm, origin='lower', cmap='viridis', vmin=-120, vmax=-40)
fig.colorbar(im, ax=ax, label='RSS [dBm]')
ax.set_title('RSS map 3.5 GHz')
plt.tight_layout()
plt.savefig('figs/ch06/fig_ch06_rss_readable.pdf', dpi=300)

## 10. アニメーションを作る

**注記**: 描画そのものは本文では省いている。`ani.save` には ffmpeg が必要。

In [ ]:
from matplotlib.animation import FuncAnimation
from sionna.rt import PathSolver
solver = PathSolver()
fig, ax = plt.subplots()
def update(t):
    # 時刻 t に応じて受信機位置を更新し、CIRを再計算
    rx_pos = trajectory(t)
    scene.remove('rx0')
    scene.add(Receiver('rx0', position=rx_pos))
    paths = solver(scene, max_depth=3)
    ax.cla()
    # ここで CIR をプロット ...
ani = FuncAnimation(fig, update, frames=60, interval=100)
ani.save('figs/ch06/anim_mobile_rx.mp4')